# 03 — Sequence structure

## Manuscript crosswalk

- **Methods:** Sequence structure analysis; transition probabilities, bigram distributions, phee-repeat distributions, and Smith–Waterman local alignment; descriptive embeddings.
- **Results:** sequence composition; eligible repertoire and comparison counts; descriptive sequence-space patterns.
- **Figures:** Figure 2 (sequence composition), Figure 4 (transition-probability repertoire space), and Figures S2–S4 (bigram, local-alignment, and phee-repeat spaces).

This notebook validates the four versioned sequence representations and their reader-facing figures. It does not recalculate local alignments or refit any embedding.

## Cached artifacts and scientific roles

- `sequence_representations.npz` stores the three vector representations in a fixed repertoire order.
- `sequence_distances.npz` stores the SciPy-order condensed local-alignment vector in the same repertoire order; this notebook expands it only for inexpensive integrity checks.
- `sequence_repertoire_distances.csv` stores model-facing distances between the two members of each focal pair’s eligible session repertoires.
- `embeddings/*.csv` stores one pooled two-dimensional fit per metric. Stage-by-context views reuse these coordinates.

The exact definitions, including local-alignment scoring and normalization, are in [the canonical analysis specification](../docs/analysis_specification.md). The same cheap plotting path is available without Jupyter through [`scripts/regenerate_cached_figures.py`](../scripts/regenerate_cached_figures.py).

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.spatial.distance import squareform


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside a clone containing README.md and data/."
    )


ROOT = find_repo_root()
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from marmoset_convergence.cli import validate_cache
from marmoset_convergence.figures import (
    EMBEDDING_FIGURES,
    prepare_embedding_table,
    regenerate_embedding_space,
    regenerate_sequence_summary,
)
from marmoset_convergence.provenance import CacheManifest

EXPENSIVE_STEPS = {
    "recompute_sequence": False,
    "recompute_dtw": False,
    "retrain_vae": False,
    "refit_models": False,
}
if any(EXPENSIVE_STEPS.values()):
    raise RuntimeError(
        "This notebook is cached-only. Expensive work requires an explicit full-pipeline command."
    )

pd.DataFrame({"step": EXPENSIVE_STEPS.keys(), "enabled": EXPENSIVE_STEPS.values()})

In [ ]:
MANIFEST_PATHS = (
    ROOT / "data/processed/manifest.json",
    ROOT / "data/cache/manifest.json",
    ROOT / "data/derived/manifest.json",
    ROOT / "results/manifest.json",
)
FIGURE_PROVENANCE_PATH = ROOT / "results/figures/figure_provenance.csv"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_manifests() -> list[tuple[Path, dict]]:
    manifests = []
    for path in MANIFEST_PATHS:
        if path.is_file():
            with path.open(encoding="utf-8") as handle:
                manifests.append((path, json.load(handle)))
    if not manifests:
        raise FileNotFoundError("No JSON provenance manifest was found.")
    return manifests


def iter_records(payload: dict):
    for section_name in ("artifacts", "sources", "inputs", "files"):
        section = payload.get(section_name, {})
        if isinstance(section, dict):
            for key, value in section.items():
                record = value if isinstance(value, dict) else {"sha256": value}
                yield str(record.get("path", key)), record
        elif isinstance(section, list):
            for record in section:
                if isinstance(record, dict) and record.get("path"):
                    yield str(record["path"]), record


def record_sha256(record: dict) -> str | None:
    value = record.get("sha256") or record.get("checksum_sha256")
    if value:
        return str(value).removeprefix("sha256:")
    checksum = record.get("checksum")
    if isinstance(checksum, str):
        return checksum.removeprefix("sha256:")
    if isinstance(checksum, dict) and checksum.get("algorithm", "").lower() == "sha256":
        return checksum.get("value")
    return None


MANIFESTS = load_manifests()
FIGURE_PROVENANCE = (
    pd.read_csv(FIGURE_PROVENANCE_PATH) if FIGURE_PROVENANCE_PATH.is_file() else pd.DataFrame()
)


def registered_artifact(relative_path: str) -> tuple[Path, dict]:
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Required cached artifact is missing: {relative_path}. No expensive fallback will run."
        )
    normalized = Path(relative_path).as_posix()
    for manifest_path, payload in MANIFESTS:
        for recorded_path, record in iter_records(payload):
            recorded = Path(recorded_path)
            same_path = (recorded.resolve() == path.resolve()) if recorded.is_absolute() else (recorded.as_posix().lstrip("./") == normalized)
            if same_path:
                expected = record_sha256(record)
                if not expected:
                    raise RuntimeError(f"No SHA-256 for {relative_path} in {manifest_path}.")
                if sha256_file(path).lower() != expected.lower():
                    raise RuntimeError(f"Checksum mismatch for {relative_path}.")
                return path, record
    if not FIGURE_PROVENANCE.empty and {"path", "sha256"}.issubset(FIGURE_PROVENANCE.columns):
        match = FIGURE_PROVENANCE.loc[FIGURE_PROVENANCE["path"] == normalized]
        if len(match) == 1:
            expected = str(match.iloc[0]["sha256"])
            if sha256_file(path).lower() != expected.lower():
                raise RuntimeError(f"Figure provenance checksum mismatch for {relative_path}.")
            return path, match.iloc[0].to_dict()
    raise RuntimeError(f"{relative_path} is not registered in a provenance manifest.")


def require_parameter(record: dict, name: str, expected: object) -> None:
    parameters = record.get("parameters")
    if not isinstance(parameters, dict) or name not in parameters:
        raise RuntimeError(f"Manifest does not record required parameter {name!r}.")
    observed = parameters[name]
    if isinstance(expected, float):
        matches = np.isclose(float(observed), expected)
    else:
        matches = observed == expected
    if not matches:
        raise RuntimeError(f"Manifest parameter {name}={observed!r}; expected {expected!r}.")

## Validate representations and the canonical repertoire order

Transition-probability and bigram vectors each use the manifest’s fixed call-type order. The phee-repeat representation contains the prespecified run-length categories. Local alignment uses match = +2, mismatch = −1, and gap = −1; it divides each sequence-pair score by the log of combined sequence lengths, averages all cross-repertoire sequence pairs, and subtracts the result from the global eligible-set maximum similarity.

In [ ]:
sequences_path, _ = registered_artifact("data/processed/sequences.csv")
inventory_path, _ = registered_artifact("data/cache/sequence_session_inventory.csv")
representations_path, representation_record = registered_artifact("data/cache/sequence_representations.npz")
distances_path, alignment_record = registered_artifact("data/cache/sequence_distances.npz")

sequences = pd.read_csv(sequences_path)
inventory = pd.read_csv(inventory_path)
required_inventory = {"repertoire_id", "n_sequences", "pair_id", "stage", "context"}
if missing := sorted(required_inventory.difference(inventory.columns)):
    raise ValueError(f"Sequence inventory is missing columns: {missing}")
if inventory["repertoire_id"].duplicated().any():
    raise ValueError("Sequence inventory contains duplicate repertoire_id values.")
expected_ids = inventory["repertoire_id"].astype(str).to_numpy()

with np.load(representations_path, allow_pickle=False) as archive:
    required_keys = {"repertoire_ids", "transition_probability", "bigram", "phee_repeat"}
    if missing := sorted(required_keys.difference(archive.files)):
        raise ValueError(f"Sequence representation cache lacks NPZ keys: {missing}")
    representation_ids = archive["repertoire_ids"].astype(str)
    transition_probability = archive["transition_probability"]
    bigram = archive["bigram"]
    phee_repeat = archive["phee_repeat"]

if not np.array_equal(representation_ids, expected_ids):
    raise ValueError("Representation IDs do not exactly match the sequence-inventory order.")
for name, expected in {
    "alphabet": ["A", "K", "P", "R", "S", "T"],
    "transition_normalization": "row_conditional",
    "bigram_normalization": "global_total",
    "phee_token": "A",
    "repeat_min": 2,
    "repeat_max": 5,
    "longer_runs": "exclude",
}.items():
    require_parameter(representation_record, name, expected)

expected_shapes = {
    "transition_probability": (107, 36),
    "bigram": (107, 36),
    "phee_repeat": (107, 4),
}
observed_arrays = {
    "transition_probability": transition_probability,
    "bigram": bigram,
    "phee_repeat": phee_repeat,
}
for name, array in observed_arrays.items():
    if array.shape != expected_shapes[name]:
        raise ValueError(f"Unexpected {name} shape: {array.shape}; expected {expected_shapes[name]}.")
    if not np.isfinite(array).all() or (array < 0).any():
        raise ValueError(f"{name} contains invalid values.")

with np.load(distances_path, allow_pickle=False) as archive:
    required_keys = {"repertoire_ids", "local_alignment"}
    if missing := sorted(required_keys.difference(archive.files)):
        raise ValueError(f"Sequence distance cache lacks NPZ keys: {missing}")
    distance_ids = archive["repertoire_ids"].astype(str)
    local_alignment_condensed = archive["local_alignment"]

if not np.array_equal(distance_ids, expected_ids):
    raise ValueError("Distance-matrix IDs do not exactly match the sequence-inventory order.")
if local_alignment_condensed.shape != (5_671,):
    raise ValueError(f"Unexpected local-alignment condensed shape: {local_alignment_condensed.shape}")
if not np.isfinite(local_alignment_condensed).all() or (local_alignment_condensed < 0).any():
    raise ValueError("Condensed local-alignment distances must be finite and non-negative.")
local_alignment = squareform(local_alignment_condensed, checks=True)
if not np.allclose(local_alignment, local_alignment.T, equal_nan=False):
    raise ValueError("The local-alignment distance matrix is not symmetric.")
if not np.allclose(np.diag(local_alignment), 0):
    raise ValueError("The local-alignment distance diagonal is not zero.")

for name, expected in {
    "algorithm": "smith_waterman",
    "match": 2,
    "mismatch": -1,
    "gap": -1,
    "normalization": "log_combined_lengths",
    "aggregation": "mean_all_cross_repertoire_pairs",
    "distance_transform": "global_max_minus_similarity",
    "storage": "condensed_scipy",
}.items():
    require_parameter(alignment_record, name, expected)
if "global_max_similarity" not in alignment_record.get("parameters", {}):
    raise RuntimeError("The local-alignment manifest lacks global_max_similarity.")

representation_audit = pd.DataFrame([
    {"representation": key, "expected_shape": value,
     "observed_shape": observed_arrays[key].shape, "passes": observed_arrays[key].shape == value}
    for key, value in expected_shapes.items()
] + [{
    "representation": "local_alignment (condensed)", "expected_shape": (5671,),
    "observed_shape": local_alignment_condensed.shape, "passes": local_alignment_condensed.shape == (5671,)
}])
display(representation_audit)

## Validate repertoire comparisons and cached embeddings

The first three repertoire metrics use Euclidean distance between their feature vectors. The alignment metric is already a repertoire distance. Every model-facing comparison remains in the original metric space.

In [ ]:
repertoire_distance_path, _ = registered_artifact(
    "data/derived/sequence_repertoire_distances.csv"
)
sequence_pair_path, _ = registered_artifact("data/cache/sequence_session_pairs.csv")
repertoire_distances = pd.read_csv(repertoire_distance_path)
sequence_pair_index = pd.read_csv(sequence_pair_path)
required = {"comparison_id", "context", "metric", "distance", "repertoire_a_id", "repertoire_b_id"}
if missing := sorted(required.difference(repertoire_distances.columns)):
    raise ValueError(f"Sequence-distance table is missing columns: {missing}")
if not np.isfinite(repertoire_distances["distance"]).all() or (repertoire_distances["distance"] < 0).any():
    raise ValueError("Sequence repertoire distances must be finite and non-negative.")
pair_key = ["comparison_id", "repertoire_a_id", "repertoire_b_id"]
if missing := sorted(set(pair_key).difference(sequence_pair_index.columns)):
    raise ValueError(f"Sequence pair index is missing columns: {missing}")
if sequence_pair_index["comparison_id"].duplicated().any():
    raise ValueError("Sequence pair index contains duplicate comparison IDs.")
distance_pairs = repertoire_distances[pair_key].drop_duplicates()
if distance_pairs["comparison_id"].duplicated().any():
    raise ValueError("One sequence comparison maps to multiple repertoire pairs.")
sequence_pair_audit = sequence_pair_index[pair_key].merge(
    distance_pairs, on=pair_key, how="outer", indicator=True, validate="one_to_one"
)
if not sequence_pair_audit["_merge"].eq("both").all():
    raise AssertionError("Sequence repertoire distances do not exactly cover the registered pair index.")

metric_aliases = {
    "transition_probability": "transition_probability",
    "transition_matrix": "transition_probability",
    "bigram": "bigram",
    "bigram_distribution": "bigram",
    "phee_repeat": "phee_repeat",
    "repeat_distribution": "phee_repeat",
    "local_alignment": "local_alignment",
}
repertoire_distances["metric_canonical"] = (
    repertoire_distances["metric"].astype(str).str.lower().str.replace(" ", "_").map(metric_aliases)
)
if repertoire_distances["metric_canonical"].isna().any():
    unknown = sorted(repertoire_distances.loc[repertoire_distances["metric_canonical"].isna(), "metric"].unique())
    raise ValueError(f"Unknown sequence metrics: {unknown}")

unique_comparisons = repertoire_distances.drop_duplicates("comparison_id")
context = unique_comparisons["context"].astype(str).str.lower().str.replace("_", "-")
context = context.replace({"stranger": "non-partner", "nonpartner": "non-partner"})

cache_manifest = CacheManifest.load(ROOT / "data/cache/manifest.json")
embedding_rows = []
sequence_embeddings = {}
for metric in ("transition_probability", "bigram", "local_alignment", "phee_repeat"):
    relative_path = f"data/cache/embeddings/{metric}.csv"
    path, record = registered_artifact(relative_path)
    manifest_names = [name for name, artifact in cache_manifest.artifacts.items() if artifact.path == relative_path]
    if len(manifest_names) != 1:
        raise ValueError(f"Expected one cache-manifest record for {relative_path}; found {len(manifest_names)}.")
    validate_cache(ROOT, manifest_names)
    table = pd.read_csv(path)
    prepare_embedding_table(table, inventory, id_column="repertoire_id")
    sequence_embeddings[metric] = table
    embedding_rows.append({
        "embedding": metric, "rows": len(table), "columns": table.shape[1],
        "method": record["parameters"]["algorithm"],
        "package": record["parameters"]["package"],
        "seed": record["parameters"]["random_seed"],
        "fit_scope": record["parameters"]["fit_scope"],
    })

count_audit = pd.DataFrame([
    {"quantity": "Eligible sequence repertoires", "expected": 107,
     "observed": inventory["repertoire_id"].nunique()},
    {"quantity": "Sequences in eligible repertoires", "expected": 1496,
     "observed": int(inventory["n_sequences"].sum())},
    {"quantity": "Unique sequence comparisons", "expected": 331,
     "observed": unique_comparisons["comparison_id"].nunique()},
    {"quantity": "Partner sequence comparisons", "expected": 62,
     "observed": int((context == "partner").sum())},
    {"quantity": "Non-partner sequence comparisons", "expected": 269,
     "observed": int((context == "non-partner").sum())},
    {"quantity": "Sequence metric rows", "expected": 1324,
     "observed": len(repertoire_distances)},
])
count_audit["passes"] = count_audit["expected"].eq(count_audit["observed"])
display(count_audit)
display(pd.DataFrame(embedding_rows))
if not count_audit["passes"].all():
    raise AssertionError("Sequence-distance manuscript count audit failed.")
if set(repertoire_distances["metric_canonical"]) != set(metric_aliases.values()):
    raise AssertionError("The distance table does not contain all four sequence metrics.")

## Figures 2, 4, and S2–S4

Figure 2 is regenerated directly from the 1,619 checksum-validated processed sequences. Figure 4 and the supplementary panels are regenerated from the versioned pooled coordinates with one fixed set of axis limits. The unsuffixed PNGs remain provenance-locked visual references; regenerated analytical exports use `_regenerated.png` names and omit the locked repertoire-card annotations. Quantitative reproduction continues to depend on the caches validated above.

In [ ]:
figure_paths = {
    "Figure 2 — sequence composition": "results/figures/main/figure_2.png",
    "Figure 4 — transition-probability space": "results/figures/main/figure_4.png",
    "Figure S2 — bigram space": "results/figures/supplement/figure_s2_bigram.png",
    "Figure S3 — local-alignment space": "results/figures/supplement/figure_s3_local_alignment.png",
    "Figure S4 — phee-repeat space": "results/figures/supplement/figure_s4_phee_repeat.png",
}
for label, relative_path in figure_paths.items():
    if (ROOT / relative_path).is_file():
        verified_path, _ = registered_artifact(relative_path)
        display(f"{label} — locked manuscript reference", Image(filename=str(verified_path)))
    else:
        print(f"{label}: optional manuscript-reference export is not present at {relative_path}.")

figure_2_regenerated = regenerate_sequence_summary(
    sequences, ROOT / figure_paths["Figure 2 — sequence composition"],
    overwrite_regenerated=True,
)
display("Figure 2 — regenerated analytical export", Image(filename=str(figure_2_regenerated)))
embedding_figure_codes = {
    "Figure 4 — transition-probability space": "4",
    "Figure S2 — bigram space": "s2",
    "Figure S3 — local-alignment space": "s3",
    "Figure S4 — phee-repeat space": "s4",
}
for label, code in embedding_figure_codes.items():
    spec = EMBEDDING_FIGURES[code]
    regenerated_path = regenerate_embedding_space(
        sequence_embeddings[spec.metric], inventory, spec,
        ROOT / spec.locked_relative_path, overwrite_regenerated=True,
    )
    display(f"{label} — regenerated analytical export", Image(filename=str(regenerated_path)))

## Result connection

The original-space model estimates are reported in Notebook 05. The cached sequence projections provide a descriptive view of the same repertoires: the manuscript reports a reduction in bonded-pair separation after pairing in the Non-partner context for transition probability, bigram, and local alignment, while the phee-repeat representation differs. These two-dimensional patterns are not independent statistical tests.